# PayChat Multi-Intent Model — Colab Training

Fine-tunes DistilBERT on the 5-intent dataset (money / alarm / contact / calendar / maps).

**Before running:** make sure the runtime is GPU — `Runtime → Change runtime type → T4 GPU` (or any GPU).

**What to upload (drag onto the Colab file panel on the left, OR run the upload cell below):**
1. `train.py`
2. `train.json`
3. `val.json`
4. `test.json`

All 4 files live in your repo at `paychat-model/training/`. Just drag them straight into Colab's file browser.

Expected run time on T4: **~10 minutes**. Final output: a zipped `saved_model.zip` downloaded to your computer.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

If the above says "command not found" or "No devices were found", you're on a CPU runtime — go to `Runtime → Change runtime type → GPU` and re-run.

## 2. Install dependencies

In [ ]:
!pip install -q "torch>=2.2.0" "transformers>=4.39.0" "scikit-learn" "numpy"

## 3. Upload the 4 training files

Run the cell below — it pops a file picker. Select **all 4 files at once**:
- `train.py`
- `train.json`
- `val.json`
- `test.json`

(You can also skip this cell and just drag the files into the left sidebar.)

In [ ]:
from google.colab import files
uploaded = files.upload()
print('\nUploaded:', list(uploaded.keys()))

## 4. Verify files are in place

In [ ]:
import os, json
required = ['train.py', 'train.json', 'val.json', 'test.json']
missing = [f for f in required if not os.path.exists(f)]
assert not missing, f'Missing files: {missing}'

for f in ['train.json', 'val.json', 'test.json']:
    with open(f) as fh:
        data = json.load(fh)
    print(f'{f:12s} -> {len(data):>5} examples')

## 5. Train

This runs the full training script. On a T4 GPU, 5 epochs over ~3400 training examples takes ~8–12 minutes. You'll see per-epoch loss + accuracy, then per-intent precision/recall/F1 on the test set.

In [ ]:
!python train.py --data-dir . --output-dir ./saved_model --epochs 5

## 6. Inspect the training report

In [ ]:
import json
with open('saved_model/training_report.json') as f:
    report = json.load(f)

print(f"Test exact-match: {report['test_exact_match']:.2%}")
print(f"Test hamming:     {report['test_hamming']:.2%}\n")

print('Per-intent F1:')
for intent, m in report['per_intent'].items():
    print(f"  {intent:<10}  P={m['precision']:.2%}  R={m['recall']:.2%}  F1={m['f1']:.2%}  AUC={m['auc']:.4f}  (support={m['support']})")

## 7. Zip and download the model

Produces `saved_model.zip` — unzip it into your repo's `paychat-model/saved_model/` (overwrite the existing one), commit, and push. The running server picks it up after `POST /reload`.

In [ ]:
!cd saved_model && zip -r ../saved_model.zip . && cd ..
!ls -lh saved_model.zip

from google.colab import files
files.download('saved_model.zip')

## 8. (optional) Quick smoke test of the trained model on sample inputs

In [ ]:
import torch, numpy as np
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

INTENTS = ['money', 'alarm', 'contact', 'calendar', 'maps']
tok = DistilBertTokenizerFast.from_pretrained('saved_model')
model = DistilBertForSequenceClassification.from_pretrained('saved_model').eval().cuda()

def predict(text):
    enc = tok(text, max_length=128, padding='max_length', truncation=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = model(**enc).logits[0].cpu().numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))
    return {intent: round(float(p), 3) for intent, p in zip(INTENTS, probs)}

samples = [
    'you owe me $40 from the airbnb',
    'remind me to take meds at 10pm',
    'hey akash save this number +1 415-555-1234',
    'meeting at 3pm tomorrow',
    'heading to dolores park',
    'remind me to venmo priya $25 at 8pm',
    'whats up',
    'that shirt was $50',
]
for s in samples:
    print(f'{s!r:55s} -> {predict(s)}')